```28/07/2025 - Gabriel A. Amici```

# Implementando classes para a biblioteca ```emerald```

### Objetivos:
- Compactar código
- Reutilizar rotinas similares
- Facilitar uso da biblioteca

### Classes base:
- ```Potential```
- ```Field```
- ```Particle```
- ```System```

In [ ]:
import numpy as np
import warnings

In [ ]:


class Potential:
    """
    Base class for 1D potentials, responsible for calculating the potential 
    and its derivatives, as well as the classical return points.
    """

    def __init__(self, *args, **kwargs):
        self.name    = None
        self.acronym = None

    def __call__(self, positions):
        return self.value(positions)
    
    def value(self, positions):
        """
        Potential evaluated at a range of positions

        Parameters
        ----------
        positions : array-like
            Array of positions at which to evaluate the potential

        Returns
        -------
        potential_values : array-like
            Array of potential values corresponding to the input positions
        """
        r = np.asarray(positions)
        pass
    
    def first_derivative(self, positions):
        """
        1st derivative of the potential evaluated for a range of positions

        Parameters
        ----------
        positions : array-like
            Array of positions at which to evaluate the derivative

        Returns
        -------
        first_derivative_values : array-like
            Array of first derivative values corresponding to the input positions
        """
        r = np.asarray(positions)
        pass
    
    def second_derivative(self, positions):
        """
        2nd derivative of the potential evaluated for a range of positions

        Parameters
        ----------
        positions : array-like
            Array of positions at which to evaluate the second derivative

        Returns
        -------
        second_derivative_values : array-like
            Array of second derivative values corresponding to the input positions
        """
        r = np.asarray(positions)
        pass
    
    def return_points(self):
        """Classical return points of the potential"""
        pass

A partir da classe base é possível implementar potenciais como classes herdadas:

In [ ]:
class MorseSoftCoulomb(Potential):
    """
    Morse-soft-Coulomb potential (MsC) is a 1D soft potential with a Morse 
    barrier at the origin. The parameter `alpha` controls the well depth and
    barrier hardness.
    """
    
    def __init__(self, alpha):
        super().__init__()
        self.name       = "Morse-soft-Coulomb"
        self.acronym    = "MsC"
        self.alpha      = alpha
        self.well_depth = 1 / alpha
        self.beta       = 1 / (alpha * np.sqrt(2))
    
    def value(self, positions):
        """
        Evaluates the Morse-soft-Coulomb potential at every point in 
        `positions`.
        """
        r = np.asarray(positions)
        
        coulomb = -1 / np.sqrt(r**2 + self.alpha**2)
        morse   = self.well_depth * (np.exp(-2 * self.beta * r) - 2 * np.exp(-self.beta * r))
        
        return np.where(r > 0, coulomb, morse)
    
    def first_derivative(self, positions):
        """
        Evaluates the first derivative of the MsC potential at the spacial points
        in `position-array`
        """
        r = np.asarray(positions)
        
        coulomb = r*(r**2 + self.alpha**2)**(-3/2)
        morse   = 2*self.well_depth*self.beta*np.exp( -2*self.beta*r)*( np.exp( self.beta*r ) - 1 )
        
        return np.where(r > 0, coulomb, morse)
    
    def second_derivative(self, positions):
        """
        Evaluates the second derivative of the MsC potential at the spacial points
        in `position-array`
        """
        r = np.asarray(positions)
        
        coulomb = (r**2 + self.alpha**2)**(-3/2) - 3*r**2*(r**2 + self.alpha**2)**(-5/2)
        morse = 4*self.well_depth*self.beta**2*np.exp( -2*self.beta*r )*( np.exp( self.beta*r ) - 0.5 )
        
        return np.where(r > 0, coulomb, morse)
    
    def return_points(self, energy: float = 0.):
        """
        Calculates the return points of the MsC potential at a given energy, 
        defaults to `(None, None)`.  
        """
        if energy < -self.well_depth:
            warnings.warn(f"Energy must be greater than the well depth ({-self.well_depth})")
            return (None, None) 

        rm = -self.alpha*np.sqrt(2)*np.log( np.sqrt( self.alpha*energy + 1 ) + 1 )
        rM = np.sqrt( 1/(energy**2) - self.alpha**2 )
        
        if energy >= 0:
            return (rm, None)
        else:
            return (rm, rM)


In [ ]:
msc = MorseSoftCoulomb(alpha=0.5)

In [ ]:
msc(1)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
r = np.linspace(-2, 15, 1000)

plt.title(f"{msc.name} Potential and its Derivatives")
plt.plot(r, msc.value(r), label='Potential')
plt.plot(r, msc.first_derivative(r), label='1st Derivative')
plt.plot(r, msc.second_derivative(r), label='2nd Derivative')
plt.xlim(-2, 15)
plt.ylim(-5, 3)
plt.legend()
plt.show()

### `Field` class

We define a field as a callable that returns the (electric) field value at time $t$. We may write a generic field object which interpolates
an array of times with an array of field values, later implementing specific fields with their corresponding parameters and analytical 
expressions:

In [ ]:
class Field:
    """
    Basic (electric) field class, given an array of field values and times, calling it results in an
    evaluation of the field (interpolated when necessary) at the specified time
    """

    def __init__(self):
        pass
    
    def __call__(self, times):
        return self.value(times)

    def value(self, times):
        """
        Evaluate the field at `times`
        """

In [ ]:
class InterpolatedField(Field):

    def __init__(self, field_times: np.ndarray, field_vals: np.ndarray):
        super().__init__()
        if len(field_times) != len(field_vals):
            raise Warning(f"Time array and field array must be of same size! "
                          f"{len(field_times)} ≠ {len(field_vals)}")

        self.field_times = field_times
        self.field_vals  = field_vals

    def value(self, times):
        """
        Evaluate the field at `times` interpolating `field_vals` and `field_times`
        """
        t = np.asarray(times)
        return np.interp(t, self.field_times, self.field_vals)

In [ ]:
class SinusoidField(Field):

    def __init__(self, amplitude: float, ang_frequency: float, phase: float):
        super().__init__()
        self.amplitude     = amplitude
        self.ang_frequency = ang_frequency
        self.phase         = phase

    def value(self, times):
        return self.amplitude*np.sin( self.ang_frequency*times + self.phase )

In [ ]:
t = np.linspace(-3, 3, 30)
F = np.sin(t)

In [ ]:
fld = InterpolatedField(t, F)

In [ ]:
times = np.linspace(-20, 20, 1000)

plt.plot(times, fld(times))

In [ ]:
F2 = SinusoidField(2, 3, np.pi/2)

plt.plot(times, F2(times))

## 3. Particle (electron)

In [ ]:
class Dynamics():

    def __init__(self, potential: Potential,
                 field: Field = None):

        self.potential = potential
        self.field     = field

    def dstate(self, state: np.ndarray = None, time: float = 0.0):
        r, p = state
        
        f1 = p
        f2 = -self.potential.first_derivative(r) - self.field(time)
        
        return np.array([f1, f2])

class StaticDynamics(Dynamics):

    def __init__(self, potential):
        super().__init__(potential)

    def dstate(self, state: np.ndarray = None, time: float = 0.0):
        r, p = state

        f1 = p
        f2 = -self.potential.first_derivative(r)
        
        return np.array([f1, f2])

In [ ]:
class Integrator:

    def __init__(self, dynamics: Dynamics, dt: float = 1.e-4):
        self.dynamics = dynamics
        self.dt       = dt

    def step(self, time: float = 0.0, state: np.ndarray = None):
        """
        Naivest integrator: forward Euler's method
        """

        return state + self.dynamics.dstate(state, time) * self.dt

class RK4Integrator(Integrator):

    def __init__(self, dynamics, dt = 1.e-4):
        super().__init__(dynamics, dt)

    def step(self, time: float = 0.0, state: np.ndarray = None):
        """
        Order 4 Runge-Kutta method for solving differencial equation systems
        """

        k1 = self.dynamics.dstate(state,                  time)
        k2 = self.dynamics.dstate(state + 0.5*self.dt*k1, time + 0.5*self.dt)
        k3 = self.dynamics.dstate(state + 0.5*self.dt*k2, time + 0.5*self.dt)
        k4 = self.dynamics.dstate(state + self.dt*k3,     time + self.dt)

        return state + (self.dt/6)*(k1 + 2*k2 + 2*k3 + k4)

In [ ]:
class ClassicalParticle():

    def __init__(self, potential: Potential, dynamics: Dynamics,
                 position: float, momentum: float, 
                 mass: float = 1.0):
        self.mass        = mass
        self.potential   = potential
        self.dynamics    = dynamics
        self.position    = position
        self.momentum    = momentum
        self.state       = np.array([position, momentum])

    def hamiltonian(self, position: float, momentum: float) -> float:
        """
        Hamiltonian of the particle at a given position and momentum
        """
        return (momentum**2)/(2*self.mass) + self.potential.value(position)

    def propagate(self, integrator: Integrator, time: float = 0.0):
        """
        Propagate the particle's state using the specified integrator
        """
        self.state = integrator.step(time, self.state)
        self.position, self.momentum = self.state

    def trajectory(self, integrator: Integrator, times: np.ndarray):
        """
        Propagate the particle's state over a range of times and return the trajectory
        """
        trajectory = np.zeros((len(times), 2))
        for i, t in enumerate(times):
            self.propagate(integrator, t)
            trajectory[i] = self.state.copy()
        return trajectory



In [ ]:
msc = MorseSoftCoulomb(alpha=0.1)
dynamics = StaticDynamics(potential=msc)
electron = ClassicalParticle(potential=msc, dynamics=dynamics, position=1.0, momentum=0.0)
integrator = RK4Integrator(dynamics=dynamics, dt=1.e-3)

electron.state
plt.plot(*electron.trajectory(integrator, np.linspace(0, 100, 10000)).T)
